# IHES Cube 106 — Base MLP Global Beam Search

This notebook runs the clean-room implementation from the public repository with model `1778521793`, a global beam of 1,000,000 states, and all 18 official generators. A result is accepted only after exact replay from the original puzzle state. The final two-column submission is then replay-validated for all 1,003 puzzles.


In [ ]:
from pathlib import Path
PUZZLE_ID = 106
MODEL_ID = "1778521793"
BEAM_WIDTH = 1_000_000
MAX_DEPTH = 30
PARENT_CHUNK = 250_000
INFERENCE_BATCH = 8_192
DEVICE = "cuda"
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/cheldieva-l/ihes-dual-model-beam-search'
REPOSITORY_REF = "main"
CHECKOUT = Path("/kaggle/working/ihes-dual-model-beam-search")
if CHECKOUT.exists():
    shutil.rmtree(CHECKOUT)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(CHECKOUT)],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(CHECKOUT), "--no-deps", "-q"], check=True)
sys.path.insert(0, str(CHECKOUT))
print(subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import numpy as np
import torch

from ihes_dual.assets import find_competition_assets
from ihes_dual.model import load_mlp2rb
from ihes_dual.puzzle import IHESPuzzle, load_test_state
from ihes_dual.registry import resolve_model

assets = find_competition_assets("/kaggle/input")
puzzle = IHESPuzzle.from_puzzle_info(assets.puzzle_info)
start = load_test_state(assets.test_csv, PUZZLE_ID)
model_spec = resolve_model("/kaggle/input", MODEL_ID)
if model_spec.model_id != MODEL_ID:
    raise AssertionError("the resolved checkpoint is not tied to the configured model ID")
model = load_mlp2rb(model_spec, DEVICE)
print({
    "puzzle_id": PUZZLE_ID,
    "model_id": model_spec.model_id,
    "checkpoint": model_spec.checkpoint.name,
    "epoch": model_spec.epoch,
    "beam_width": BEAM_WIDTH,
    "device": DEVICE,
    "generator_count": puzzle.generator_count,
})
assert puzzle.generator_count == 18


In [ ]:
from dataclasses import asdict
from ihes_dual.beam import BeamConfig, beam_search
from ihes_dual.solve import write_run_log
from ihes_dual.submission import build_submission, validate_submission

config = BeamConfig(
    beam_width=BEAM_WIDTH,
    max_depth=MAX_DEPTH,
    parent_chunk=PARENT_CHUNK,
    inference_batch=INFERENCE_BATCH,
    device=DEVICE,
    autocast=True,
    prune_immediate_inverse=False,
    diagnostic_protect=False,
)
trace = beam_search(puzzle, start, model, config)
if trace.solution is None:
    raise RuntimeError("base beam did not find a solution")
if not 22 <= len(trace.solution) <= 30:
    raise AssertionError(f"controlled solution length is {len(trace.solution)}, expected 22..30")
if not puzzle.verify_solution(start, trace.solution):
    raise AssertionError("base solution failed exact replay")
print("solution length:", len(trace.solution))
print("solution:", puzzle.encode_path(trace.solution))

run_dir = OUTPUT_DIR / "runs" / f"model-{MODEL_ID}" / f"puzzle-{PUZZLE_ID}" / "base"
write_run_log(
    run_dir / "run.json",
    {
        "mode": "base",
        "model_id": MODEL_ID,
        "puzzle_id": PUZZLE_ID,
        "solution": puzzle.encode_path(trace.solution),
        "solution_length": len(trace.solution),
        "replay_valid": True,
        "config": asdict(config),
        "depths": trace.diagnostic_report()["depths"],
    },
)
submission_path = build_submission(
    assets.sample_submission,
    OUTPUT_DIR / "submission.csv",
    puzzle,
    {PUZZLE_ID: trace.solution},
)
validation = validate_submission(submission_path, assets.test_csv, puzzle)
print("submission:", submission_path)
print("validation:", validation)
